
#### 🌟 主題：籃球賽事播

**🎯 功能說明：**
1. 增加賽況輸入欄位，比分、球隊名稱、關鍵事件
2. Gradio 呈現：三個欄位：初稿播報文案、修改建議、優化後播報文案




#### 1. 讀入你的金鑰

請依你使用的服務, 決定讀入哪個金鑰

In [ ]:
import os
from google.colab import userdata

In [ ]:
#使用Groq
api_key = userdata.get('Groq')
os.environ['GROQ_API_KEY']=api_key
provider = "groq"
model = "llama3-70b-8192"

In [ ]:
!pip install aisuite[all]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.9/863.9 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.5/103.5 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 45.9 MB/s eta 0:00:00
  Attempting uninstall: httpx
    Found existing installation: httpx 0.28.1
    Uninstalling httpx-0.28.1:
      Successfully uninstalled httpx-0.28.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 1.16.1 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.27.2 which is incompatible.


### 2. 基本的設定

In [ ]:
import aisuite as ai

In [ ]:
provider_writer = "groq"
model_writer="llama3-70b-8192"

provider_reviewer = "groq"
model_reviewer = "llama3-70b-8192"

#provider_reviewer = "openai"
#model_reviewer = "gpt-4o"

標準回應函式

In [ ]:
def reply(system="請用台灣習慣的中文回覆。",
          prompt="hi",
          provider="groq",
          model="llama3-70b-8192"
          ):

    client = ai.Client()

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt}
    ]


    response = client.chat.completions.create(model=f"{provider}:{model}", messages=messages)

    return response.choices[0].message.content

####  3. 人物設定

In [ ]:
system_writer = "你是一位活潑、有感染力的籃球賽事播報主播，專業且有熱情，善於用台灣習慣的中文描繪比賽精彩瞬間。請用第一人稱，讓觀眾感受到現場氣氛，有時加點幽默和 emoji。"
system_reviewer = "你是一位文案專家，擅長讓體育播報文字更口語化、生活化，讓文字生動自然。請針對以下播報文案給出具體修改建議，使用台灣習慣中文。"

In [ ]:
def reflect_basketball_report(team_a, score_a, team_b, score_b, highlight):
    # Step1:生成初稿播報文案
    prompt = (
        f"請撰寫一段籃球比賽播報文字，內容包含：\n"
        f"隊伍A：{team_a}，得分：{score_a}\n"
        f"隊伍B：{team_b}，得分：{score_b}\n"
        f"比賽亮點：{highlight}\n"
        f"請用生動且帶熱情的主播口吻撰寫。"
    )
    first_version = reply(system_writer, prompt, provider=provider, model=model_writer)

    # Step2:文案審稿給建議
    suggestion = reply(system_reviewer, first_version, provider=provider, model=model_reviewer)

    # Step3:根據建議再改寫一次
    second_prompt = (
        f"這是我剛剛寫的播報文案：\n{first_version}\n\n"
        f"這是修改建議：\n{suggestion}\n\n"
        f"請根據建議改寫文案，讓播報更流暢自然，台灣用語，且只輸出改好的文案。"
    )
    second_version = reply(system_writer, second_prompt, provider=provider, model=model_writer)

    return first_version, suggestion, second_version

### 4. 用 Gradio 打造你的籃球播報生成系統

In [ ]:
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.2/54.2 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.1/323.1 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 103.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 4.5 MB/s eta 0:00:00


In [ ]:
import gradio as gr

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("### 🏀 籃球賽事播報文案生成與優化")
    with gr.Row():
        team_a = gr.Textbox(label="隊伍 A 名稱", placeholder="輸入隊伍名稱")
        score_a = gr.Number(label="隊伍 A 得分", value=0)
        team_b = gr.Textbox(label="隊伍 B 名稱", placeholder="輸入隊伍名稱")
        score_b = gr.Number(label="隊伍 B 得分", value=0)
    highlight = gr.Textbox(label="比賽亮點（關鍵事件、精彩瞬間）", lines=3, placeholder="輸入內容")

    btn = gr.Button("生成播報文案 & 修正建議")

    with gr.Row():
        out1 = gr.Textbox(label="📝 初稿播報文案", lines=7)
        out2 = gr.Textbox(label="🔍 修改建議", lines=5)
        out3 = gr.Textbox(label="✨ 優化後播報文案", lines=7)

    btn.click(
        reflect_basketball_report,
        inputs=[team_a, score_a, team_b, score_b, highlight],
        outputs=[out1, out2, out3]
    )

In [ ]:
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://722038033705848862.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://722038033705848862.gradio.live
